# Robust NMF experiments

## Overview

This notebook provides a clean research workflow for comparing Euclidean NMF with a robust residual-based NMF under controlled salt-and-pepper corruption. Algorithm, noise, metric, data, and plotting implementations are imported from `robust_nmf`; no method is redefined here. Stored aggregate values are treated as completed experimental evidence, while optional reruns require local datasets.

## Prerequisites

From the repository root, create and activate a Python environment, then run `python -m pip install -e ".[dev,report]"`. Launch Jupyter from the repository root so imports and relative paths resolve consistently.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from robust_nmf.data import load_face_directory
from robust_nmf.metrics import evaluate_clustering, relative_reconstruction_error
from robust_nmf.nmf import fit_l2_nmf, fit_l21_nmf
from robust_nmf.noise import add_salt_pepper_noise
from robust_nmf.visualization import load_summary, plot_metric_comparison

## Configuration and data paths

All paths are relative to the repository root. The source image directories remain local and are intentionally excluded from version control. Seeds and lightweight defaults are explicit so that optional runs can be repeated.

In [ ]:
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA_PATHS = {
    'ORL': ROOT / 'data' / 'ORL',
    'Extended YaleB': ROOT / 'data' / 'CroppedYaleB',
}
SUMMARY_PATH = ROOT / 'results' / 'metrics' / 'summary.csv'
SEED = 17
CORRUPTIONS = (0.2, 0.4, 0.6)
SALT_RATIOS = (0.1, 0.7)
REPEATS = 5

## Load and audit

The audit reports availability without exposing filenames, class labels, or images. Loading occurs only when a configured directory is present.

In [ ]:
availability = {name: path.is_dir() for name, path in DATA_PATHS.items()}
availability

In [ ]:
def load_available_datasets(paths, resize_by_name):
    loaded = {}
    for name, path in paths.items():
        if path.is_dir():
            loaded[name] = load_face_directory(path, resize=resize_by_name[name])
    return loaded


datasets = load_available_datasets(
    DATA_PATHS, {'ORL': (30, 37), 'Extended YaleB': (42, 48)}
)
audit = {
    name: {
        'matrix_shape': data.matrix.shape,
        'image_shape': data.image_shape,
        'classes': len(data.class_names),
        'finite': bool(np.isfinite(data.matrix).all()),
        'range': (float(data.matrix.min()), float(data.matrix.max())),
    }
    for name, data in datasets.items()
}
audit

## Controlled corruption visualization

A synthetic grayscale pattern demonstrates the corruption controls without embedding source imagery in the notebook.

In [ ]:
synthetic_image = np.linspace(0.0, 1.0, 24 * 24).reshape(24, 24)
settings = [(0.0, 0.5), (0.2, 0.1), (0.4, 0.7), (0.6, 0.7)]
figure, axes = plt.subplots(1, len(settings), figsize=(10, 2.8))
for axis, (corruption, salt_ratio) in zip(axes, settings):
    shown = add_salt_pepper_noise(
        synthetic_image, corruption, salt_ratio, seed=SEED
    )
    axis.imshow(shown, cmap='gray', vmin=0.0, vmax=1.0)
    axis.set_title(f'noise={corruption:g}, salt={salt_ratio:g}')
    axis.axis('off')
figure.tight_layout()

## Repeated L2 versus L21 evaluation

The function below is an optional controlled rerun for an already loaded dataset. Each repeat uses a derived seed for corruption, factor initialization, and clustering. Reconstruction is evaluated against the clean matrix, and clustering uses coefficient columns. Runtime depends on local data and the chosen iteration budget.

In [ ]:
def run_repeated_evaluation(
    matrix, labels, *, rank, corruption, salt_ratio, repeats=REPEATS, max_iter=100
):
    records = []
    for repeat in range(repeats):
        run_seed = SEED + repeat
        noisy = add_salt_pepper_noise(
            matrix, corruption, salt_ratio, seed=run_seed
        )
        fits = {
            'L2-NMF': fit_l2_nmf(
                noisy, rank, max_iter=max_iter, seed=run_seed
            ),
            'L21-NMF': fit_l21_nmf(
                noisy, rank, max_iter=max_iter, seed=run_seed
            ),
        }
        for method, fit in fits.items():
            clustering = evaluate_clustering(
                fit.coefficients, labels, seed=run_seed
            )
            records.append({
                'repeat': repeat,
                'method': method,
                'rre': relative_reconstruction_error(
                    matrix, fit.reconstruction
                ),
                'accuracy': clustering.accuracy,
                'nmi': clustering.nmi,
            })
    return records

## Stored-results visualization

The checked-in summary contains rounded aggregate values from completed repeated experiments. Loading validates its schema, numeric types, ranges, and unique condition keys. These cells visualize that stored evidence; they do not rerun the source-data pipeline.

In [ ]:
summary = load_summary(SUMMARY_PATH)
summary.groupby(['dataset', 'method']).size()

In [ ]:
rre_figure = plot_metric_comparison(summary, metrics=('rre',))
clustering_figure = plot_metric_comparison(
    summary, metrics=('accuracy', 'nmi')
)

## Limitations and reproducibility

- Stored metrics are rounded aggregate values, so they cannot support higher-precision claims.
- The completed experiment used method-specific configurations and an asymmetric clean-data evaluation protocol: L2,1-NMF refit the coefficient matrix `H` on clean data while keeping `W` fixed, whereas L2-NMF metrics directly evaluated factors learned on noisy data without clean-data refitting. This makes the cross-method clustering and reconstruction comparison not a perfectly symmetric protocol, so direct causal attribution to the loss alone is limited.
- The current library uses a documented residual-based robust objective. Optional reruns from this notebook are a reproducible new protocol, not an exact recreation of the stored experiment.
- Local datasets are not distributed with the repository. Their provenance, licensing, and integrity must be verified independently.
- The smoke experiment checks numerical invariants on synthetic data and is not evidence of source-data performance.